## 8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?

Dada la matriz simétrica y definida positiva $A \in \mathbb{R}^{p \times p}$ tal que 
$$
A = LL^T
$$ 

siendo $L$ una matriz triangular inferior.

El determinante de $L$ es el producto de su diagonal, y en la factorización de Cholesky todos los elementos de la diagonal son positivos (ya que $A$ es definida positiva). Por lo tanto, $L$ tiene determinante distinto de 0 y es invertible.

Por la propiedad de la inversa de un producto de matrices:

$$
A^{-1} = (LL^T)^{-1} = (L^T)^{-1} L^{-1}
$$

y como $(L^T)^{-1} = (L^{-1})^T$, se obtiene:

$$
A^{-1} = (L^{-1})^T L^{-1}
$$


Teniendo QDA definida como:
$$
\delta_k(x) = -\frac{1}{2} \ln |\Sigma_k| - \frac{1}{2} (x - \mu_k)^T \Sigma_k^{-1} (x - \mu_k) + \ln \pi_k
$$

se necesita invertir $\Sigma_k$ para cada clase. Invertir matrices con un método genérico es computacionalmente costoso, además de numéricamente inestable. Por lo visto anterior, Cholesky permite descomponer $\Sigma_k = L_k L_k^T$, con $L_k$ triangular inferior, quedando etonces:

$$
\delta_k(x) = -\frac{1}{2} \ln |L_k L_k^T| - \frac{1}{2} (x - \mu_k)^T (L_k^{-1})^T L_k^{-1} (x - \mu_k) + \ln \pi_k
$$

Dado que $L_k$ es triangular, su determinante es el producto de su diagonal:

$$
|\Sigma_k| = |L_k L_k^T| = |L_k|^2 = \left( \prod_{i=1}^{p} (L_k)_{ii} \right)^2
$$

por lo que el primer término se reduce a:

$$
-\frac{1}{2} \ln |\Sigma_k| = -\ln \prod_{i=1}^{p} (L_k)_{ii}
$$

Por la propiedad de la traspuesta de un producto, se tiene que 
$$
(x - \mu_k)^T (L_k^{-1})^T = \left( L_k^{-1} (x - \mu_k) \right)^T
$$ 

Definiendo un vector $z_k$ como $z_k = L_k^{-1} (x - \mu_k)$, el término cuadrático queda como la norma al cuadrado de ese vector:

$$
\frac{1}{2} (x - \mu_k)^T (L_k^{-1})^T L_k^{-1} (x - \mu_k) = \frac{1}{2} z_k^T z_k = \frac{1}{2} \|z_k\|^2
$$

$z_k$ puede obtenerse resolviendo el sistema de ecuaciones lineales:

$$
L_k z_k = x - \mu_k
$$

Dado que $L_k$ es triangular inferior, este sistema se resuelve por sustitución hacia adelante: resuelvo la primer fila con el único término distinto a 0 siendo la primer componente de $z_k$, luego reemplazo en la segunda fila para que quede una sola incógnita y resuelvo para la segunda componente, y así sucesivamente. No hay necesidad de calcular ninguna matriz inversa.

La factorización de Cholesky también tiene costo $O(p^3)$, pero requiere aproximadamente un tercio de las operaciones de una inversión normal y es numéricamente más estable. Además, una vez obtenida $L_k$, el determinante y la forma cuadrática se calculan de forma mucho más barata.



## 9. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.
Ambas `QDA` y `QDA_Chol1` heredan de `BaseBayesianClassifier` y comparten un flujo general. Las diferencias están en `_fit_params` (que define qué es lo que guarda al entrenar el modelo) y `_predict_log_conditional` (que define cómo se calcula el puntaje de cada clase).

# Fit

Para cada clase k, ambos modelos seleccionan las observaciones que pertencen a esa clase y calculan la matriz de covarianza $\Sigma_k$ y su media $\mu_k$.

`QDA` calcula y almacena la inversa de cada matriz de covarianzas $\Sigma_k$.

Mientras tanto, `QDA_Chol1` utiliza `cholesky` para expresar a la matriz $\Sigma$ como el producto de una matriz triangular inferior $L_k$ y su traspuesta. Luego calcula la inversa de esta matriz $L_k$ (curiosamente, de forma genérica en vez de aprovechar su forma) para cada clase y las almacena.
En ambos casos se realiza la inversión de una matriz, la diferencia es qué matriz se invierte.

# Predict

Ambos modelos heredan un flujo base: para cada observación $x$ se recorren todas las clases y se calcula el puntaje de cada una con `_predict_log_conditional`. Se suma el logaritmo de su probabilidad y se predice la clase como aquella que tenga mayor puntaje con `argmax`.

Ambos calculan `unbiased_x = x - self.means[class_idx]` ($x - \mu_k$).

Para calcular el determinante, `QDA` lo hace de forma genérica.

`QDA_Chol1` aprovecha la forma de la matriz  $L_k$ para calcular determinantes usando solo el producto de la diagonal con `np.log(L_inv.diagonal().prod())`. Como la diagonal $L_k^{-1}$ es la diagonal de $L_k$ invertida, este calor da lo mismo que el término $-\frac{1}{2} \ln |\Sigma_k|$ de `QDA` (por eso aparece con signo positivo).

Para el término cuadrático, `QDA_Chol1` multiplica $L_k^{-1}$ por $x - \mu_k$ con `y = L_inv @ unbiased_x`, que es el vector $z_k$ del punto (8). Después calcula su norma al cuadrado con `(y**2).sum()`. En cambio, `QDA` multiplica por la matriz $\Sigma_k$ inversa entera.

Ambos modelos llegan al mismo puntaje para cada clase y realizan las mismas predicciones.